# 조건부 엣지로 경로 선택

검색 점수가 기준을 넘는지에 따라 답변 생성과 추가 자료 요청 중 어느 노드로 이동하게 할까요?

## 실행 환경 준비

Python 3.12 새 커널에서 첫 셀부터 실행합니다. 설치 셀은 이 노트북에 필요한 버전을 고정합니다.

## 패키지 설치와 버전 확인

필요한 패키지를 고정 버전으로 설치합니다. 출력에서 LangGraph 버전 또는 API 키의 존재 여부만 확인합니다.

In [1]:
# 노트북을 독립 실행할 수 있도록 패키지와 환경을 준비합니다.
%pip install -q langgraph==1.2.11 langchain==1.4.0 langchain-openai==1.6.1 python-dotenv==1.2.3 scikit-learn==1.9.0
from importlib.metadata import version
print('Python 패키지 준비 완료')
print('langgraph', version('langgraph'))

Note: you may need to restart the kernel to use updated packages.
Python 패키지 준비 완료
langgraph 1.2.11


## 상태와 분기 함수 정의

다음 코드에서 상태와 분기 함수 정의에 필요한 형식과 함수를 선언합니다. 이 셀은 정의를 준비하며, 값은 이어지는 실행 셀에서 확인합니다.

In [2]:
# 상태와 분기 함수 정의에 사용할 상태 형식과 함수를 선언합니다.
from typing import Literal
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class RouteState(TypedDict):
    rerank_score: float
    route: str
    answer: str

def inspect_score(state: RouteState) -> dict:
    return {'route': 'checked'}

def choose(state: RouteState) -> Literal['generate', 'request_more']:
    return 'generate' if state['rerank_score'] >= 0.7 else 'request_more'

## 두 목적지 노드 정의

다음 코드에서 두 목적지 노드 정의에 필요한 형식과 함수를 선언합니다. 이 셀은 정의를 준비하며, 값은 이어지는 실행 셀에서 확인합니다.

In [3]:
# 두 목적지 노드 정의에 사용할 상태 형식과 함수를 선언합니다.
def generate(state: RouteState) -> dict:
    return {'route': 'generate', 'answer': '근거로 답변 생성'}

def request_more(state: RouteState) -> dict:
    return {'route': 'request_more', 'answer': '추가 자료 요청'}

builder = StateGraph(RouteState)
builder.add_node('inspect_score', inspect_score)
builder.add_node('generate', generate)
builder.add_node('request_more', request_more)

## 조건부 엣지 연결

조건부 엣지 연결 단계를 실행합니다. 출력된 상태 값과 코드에서 반환한 키를 서로 대조합니다.

In [4]:
# 조건부 엣지 연결 단계의 연산을 실행하고 반환된 상태 값을 확인합니다.
builder.add_edge(START, 'inspect_score')
builder.add_conditional_edges('inspect_score', choose)
builder.add_edge('generate', END)
builder.add_edge('request_more', END)
graph = builder.compile()
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	inspect_score(inspect_score)
	generate(generate)
	request_more(request_more)
	__end__([<p>__end__</p>]):::last
	__start__ --> inspect_score;
	inspect_score -.-> generate;
	inspect_score -.-> request_more;
	generate --> __end__;
	request_more --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 점수 하나를 바꿔 경로 비교

점수 하나를 바꿔 경로 비교 조건으로 값을 하나만 변경합니다. 나란히 출력된 두 결과에서 달라진 항목을 확인합니다.

In [5]:
# 한 조건만 바꾸고 나머지 입력을 유지해 결과 차이를 비교합니다.
base = {'route': '', 'answer': ''}
high = graph.invoke({**base, 'rerank_score': 0.82})
low = graph.invoke({**base, 'rerank_score': 0.41})
print('0.82:', high['route'], high['answer'])
print('0.41:', low['route'], low['answer'])
assert high['route'] == 'generate'
assert low['route'] == 'request_more'

0.82: generate 근거로 답변 생성
0.41: request_more 추가 자료 요청
